In [2]:
# Install dependencies
# pip install stable-baselines3[extra] gym


import gym
from stable_baselines3 import PPO
from stable_baselines3.common.evaluation import evaluate_policy

# Create the environment
env = gym.make("CartPole-v1")

# Create the model
model = PPO("MlpPolicy", env, verbose=1)

# Train the model
model.learn(total_timesteps=10_000)

# Evaluate the model
mean_reward, std_reward = evaluate_policy(model, env, n_eval_episodes=10)
print(f"Mean reward: {mean_reward:.2f} ± {std_reward:.2f}")

# Use the trained model
# Fix for Gym API vs SB3 VecEnv API issue
obs, info = env.reset()  # Gym API returns (obs, info) tuple
for _ in range(1000):
    action, _ = model.predict(obs, deterministic=True)  # Pass only the observation
    obs, reward, terminated, truncated, info = env.step(action)  # Updated Gym API
    done = terminated or truncated  # Combine terminated and truncated for done state
    env.render()
    if done:
        obs, info = env.reset()  # Get observation and info

env.close()


Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 22.1     |
|    ep_rew_mean     | 22.1     |
| time/              |          |
|    fps             | 5191     |
|    iterations      | 1        |
|    time_elapsed    | 0        |
|    total_timesteps | 2048     |
---------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 25.5        |
|    ep_rew_mean          | 25.5        |
| time/                   |             |
|    fps                  | 2620        |
|    iterations           | 2           |
|    time_elapsed         | 1           |
|    total_timesteps      | 4096        |
| train/                  |             |
|    approx_kl            | 0.008221917 |
|    clip_fraction        | 0.0921      |
|    clip_range           | 0.2         |
|    entropy_loss   

/opt/homebrew/anaconda3/envs/python-notebook/lib/python3.9/site-packages/gym/envs/classic_control/cartpole.py:211: UserWarning: WARN: You are calling render method without specifying any render mode. You can specify the render_mode at initialization, e.g. gym("CartPole-v1", render_mode="rgb_array")
  gym.logger.warn(


In [3]:
import gym
from gym import spaces
import numpy as np

class PromoPricingEnv(gym.Env):
    """
    Custom Env for reinforcement learning in promotion pricing.
    Goal: Maximize revenue or profit over N time steps.
    """
    def __init__(self):
        super(PromoPricingEnv, self).__init__()

        # Example state: [demand_forecast, remaining_budget, time_of_week]
        self.observation_space = spaces.Box(low=0, high=1, shape=(3,), dtype=np.float32)

        # Actions: [0%, 10%, 20%, 30%, 40% discount]
        self.action_space = spaces.Discrete(5)

        self.current_step = 0
        self.max_steps = 30  # e.g., simulate a month
        self.remaining_budget = 1.0

    def reset(self):
        self.current_step = 0
        self.remaining_budget = 1.0
        return self._get_obs()

    def _get_obs(self):
        demand_forecast = np.random.rand()
        time_of_week = (self.current_step % 7) / 6  # normalize
        return np.array([demand_forecast, self.remaining_budget, time_of_week], dtype=np.float32)

    def step(self, action):
        discount_levels = [0.0, 0.1, 0.2, 0.3, 0.4]
        discount = discount_levels[action]

        # Simulate demand: higher discount = more conversion
        base_demand = np.random.uniform(0.5, 1.0)
        conversion = base_demand * (1 + discount * 2)  # example formula
        unit_price = 1.0 * (1 - discount)
        revenue = conversion * unit_price

        # Budget consumption logic (can be marketing cost, e.g., cogs)
        budget_used = conversion * discount * 0.5
        self.remaining_budget = max(0.0, self.remaining_budget - budget_used)

        self.current_step += 1
        done = self.current_step >= self.max_steps or self.remaining_budget <= 0.0

        reward = revenue  # or: revenue - cost, or CLV proxy

        return self._get_obs(), reward, done, {}

    def render(self, mode="human"):
        print(f"Step: {self.current_step}, Remaining budget: {self.remaining_budget:.2f}")


In [4]:
from stable_baselines3 import PPO

env = PromoPricingEnv()
model = PPO("MlpPolicy", env, verbose=1)
model.learn(total_timesteps=50_000)


Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 9.2      |
|    ep_rew_mean     | 7.51     |
| time/              |          |
|    fps             | 6589     |
|    iterations      | 1        |
|    time_elapsed    | 0        |
|    total_timesteps | 2048     |
---------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 10.6        |
|    ep_rew_mean          | 8.51        |
| time/                   |             |
|    fps                  | 4150        |
|    iterations           | 2           |
|    time_elapsed         | 0           |
|    total_timesteps      | 4096        |
| train/                  |             |
|    approx_kl            | 0.022893626 |
|    clip_fraction        | 0.324       |
|    clip_range           | 0.2         |
|    entropy_loss   

In [17]:
model.predict(env.reset())

(array(0), None)